In [ ]:
# 1️⃣ 운영 형태 컬럼 생성
def assign_operating_type(row):
    if row['room_type'] == 'Hotel room' or 'Hotel' in str(row['property_type']):
        return '호텔'
    elif row['minimum_nights'] >= 30:
        return '중기/장기'
    else:
        if pd.notnull(row.get('license')):  # 라이센스 있는 단기
            return '단기'
        else:
            return '단기_미등록'

df['operating_type'] = df.apply(assign_operating_type, axis=1)

# 2️⃣ 추천 함수 정의
def recommend_host(data, persona, initial_investment):
    # 필터링
    filtered = data[
        (data['operating_type'] == persona['operating_type']) &
        (data['room_type'] == persona['room_type']) &
        (data['accommodates'] >= persona['accommodates_min']) &
        (data['bedrooms'] >= persona['bedrooms_min']) &
        (data['beds'] >= persona['beds_min'])
    ]
    
    if persona.get('instant_bookable'):
        filtered = filtered[filtered['instant_bookable'] == True]
    if persona.get('license_required'):
        filtered = filtered[filtered['license'].notnull()]
    
    if filtered.empty:
        print("조건에 맞는 숙소가 없습니다. 조건을 완화해보세요.")
        return pd.DataFrame()
    
    # 세그먼트별 성과 계산
    seg = filtered.groupby(['neighbourhood_cleansed','room_type']).agg(
        median_price=('price','median'),
        median_occupancy=('occupancy_rate','median'),
        median_revenue=('estimated_revenue_l365d','median'),
        listing_count=('price','count')  # id 없으면 price로 대체
    ).reset_index()
    
    # 시장 포지션
    price_median = seg['median_revenue'].median()
    supply_median = seg['listing_count'].median()
    
    def market_position(row):
        if row['listing_count'] > supply_median and row['median_revenue'] > price_median:
            return '레드오션'
        elif row['listing_count'] <= supply_median and row['median_revenue'] > price_median:
            return '블루오션'
        else:
            return '저수익'
    
    seg['market_position'] = seg.apply(market_position, axis=1)
    
    # 월 수익 & 회수기간
    seg['monthly_revenue'] = seg['median_price'] * seg['median_occupancy'] * 30
    seg['monthly_profit'] = seg['monthly_revenue'] * 0.75  # 비용 25% 가정
    seg['payback_months'] = initial_investment / seg['monthly_profit']
    
    # Top 5 추천
    top5 = seg.sort_values(by='median_revenue', ascending=False).head(5)
    
    return top5

# 3️⃣ 페르소나 입력
persona = {
    'operating_type': '중기/장기',
    'room_type': 'Entire home/apt',
    'accommodates_min': 2,
    'bedrooms_min': 1,
    'beds_min': 1,
    'instant_bookable': True,
    'license_required': True
}

initial_investment = 10000

# 4️⃣ 추천 실행
top5 = recommend_host(df, persona, initial_investment)
top5